In [7]:
"""
Project: Parameter-Efficient Federated Aggregator for Heterogeneous Edge Nodes
Author: Kehinde Julius Adediran
Scope: Production-grade implementation of Parameter-Efficient Federated Learning (PEFT-FL).
       Features Dirichlet Non-IID data partitioning, adapter-based local updates,
       weighted FedAvg aggregation, and dynamic communication bandwidth benchmarking.
"""

import argparse
import copy
from typing import Dict, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

In [9]:
# =============================================================================
# 1. ARCHITECTURE DEFINITION (ConvNet Backbone + PEFT Adapters)
# =============================================================================

class PEFTAdapterBlock(nn.Module):
    """
    Bottleneck Adapter Layer for Parameter-Efficient Fine-Tuning.
    Down-projects latent dimension to rank r, applies non-linearity, and projects back.
    """
    def __init__(self, in_features: int, bottleneck_dim: int):
        super().__init__()
        self.down_proj = nn.Linear(in_features, bottleneck_dim, bias=False)
        self.act = nn.ReLU()
        self.up_proj = nn.Linear(bottleneck_dim, in_features, bias=False)
        
        # Zero-initialize up-projection so adapter initially acts as an identity mapping
        nn.init.zeros_(self.up_proj.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.up_proj(self.act(self.down_proj(x)))


class DeepAgentPolicyNetwork(nn.Module):
    """
    Feature Extractor (CNN) with injected Bottleneck PEFT Adapters and Policy Head.
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 10, adapter_rank: int = 16):
        super().__init__()
        
        # Deep Feature Extractor Backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        
        self.flatten_dim = 64 * 7 * 7  # 28x28 spatial inputs
        
        # Latent Projection Layer
        self.fc_latent = nn.Linear(self.flatten_dim, 256)
        
        # PEFT Bottleneck Adapter
        self.adapter = PEFTAdapterBlock(in_features=256, bottleneck_dim=adapter_rank)
        
        # Task / Policy Head
        self.head = nn.Linear(256, num_classes)

    def freeze_backbone(self) -> None:
        """Freezes representation backbone, making only Adapters and Head trainable."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.fc_latent.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        latent = F.relu(self.fc_latent(x))
        
        # Residual adapter injection
        adapted = latent + self.adapter(latent)
        logits = self.head(adapted)
        return logits


# =============================================================================
# 2. DATA PARTITIONING (Dirichlet Non-IID Distribution)
# =============================================================================

def partition_data_dirichlet(
    dataset: Dataset, num_clients: int, alpha: float = 0.5
) -> List[Subset]:
    """
    Partitions a dataset into Non-IID client subsets using a Dirichlet distribution over class labels.
    """
    if hasattr(dataset, 'targets'):
        labels = np.array(dataset.targets)
    elif hasattr(dataset, 'labels'):
        labels = np.array(dataset.labels)
    else:
        labels = np.array([y for _, y in dataset])

    num_classes = len(np.unique(labels))
    label_indices = [np.where(labels == i)[0] for i in range(num_classes)]
    
    client_indices = [[] for _ in range(num_clients)]
    
    for k in range(num_classes):
        idx_k = label_indices[k]
        np.random.shuffle(idx_k)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        
        # Split indices based on proportions
        proportions = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
        splits = np.split(idx_k, proportions)
        
        for i, split in enumerate(splits):
            client_indices[i].extend(split)
            
    return [Subset(dataset, idxs) for idxs in client_indices]


# =============================================================================
# 3. FEDERATED AGENT NODE & CENTRAL SERVER
# =============================================================================

class FederatedAgentNode:
    """Decentralized edge client running local PEFT training passes."""
    
    def __init__(
        self,
        node_id: int,
        model_template: nn.Module,
        subset: Subset,
        batch_size: int = 32,
        lr: float = 0.01,
        device: str = "cpu",
    ):
        self.node_id = node_id
        self.device = device
        self.local_model = copy.deepcopy(model_template).to(self.device)
        self.data_loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
        self.num_samples = len(subset)
        
        # Enforce PEFT constraints
        self.local_model.freeze_backbone()
        
        trainable_params = [p for p in self.local_model.parameters() if p.requires_grad]
        self.optimizer = optim.SGD(trainable_params, lr=lr, momentum=0.9)
        self.criterion = nn.CrossEntropyLoss()

    def train_epoch(self, epochs: int = 2) -> Tuple[Dict[str, torch.Tensor], int]:
        self.local_model.train()
        for _ in range(epochs):
            for data, targets in self.data_loader:
                data, targets = data.to(self.device), targets.to(self.device)
                self.optimizer.zero_grad()
                logits = self.local_model(data)
                loss = self.criterion(logits, targets)
                loss.backward()
                self.optimizer.step()
                
        # Extract strictly trainable parameter tensors (Adapters + Head)
        trainable_state_dict = {
            name: param.cpu().detach().clone()
            for name, param in self.local_model.named_parameters()
            if param.requires_grad
        }
        return trainable_state_dict, self.num_samples


class FederatedServer:
    """Central server executing weighted FedAvg and global benchmark evaluations."""
    
    def __init__(self, global_model: nn.Module, device: str = "cpu"):
        self.global_model = global_model.to(device)
        self.device = device

    @torch.no_grad()
    def aggregate_updates(self, client_updates: List[Tuple[Dict[str, torch.Tensor], int]]) -> None:
        total_samples = sum(n for _, n in client_updates)
        if total_samples == 0:
            raise ValueError("Zero total samples encountered across clients.")

        global_dict = self.global_model.state_dict()
        update_keys = client_updates[0][0].keys()
        
        accumulators = {k: torch.zeros_like(global_dict[k], dtype=torch.float32) for k in update_keys}
        
        for client_dict, num_samples in client_updates:
            weight = num_samples / total_samples
            for key in update_keys:
                accumulators[key] += client_dict[key].to(dtype=torch.float32) * weight
                
        for key in update_keys:
            global_dict[key].copy_(accumulators[key])
            
        self.global_model.load_state_dict(global_dict)

    @torch.no_grad()
    def evaluate(self, test_loader: DataLoader) -> Tuple[float, float]:
        self.global_model.eval()
        criterion = nn.CrossEntropyLoss()
        total_loss, correct, total = 0.0, 0, 0
        
        for data, targets in test_loader:
            data, targets = data.to(self.device), targets.to(self.device)
            logits = self.global_model(data)
            loss = criterion(logits, targets)
            
            total_loss += loss.item() * data.size(0)
            preds = logits.argmax(dim=1)
            correct += preds.eq(targets).sum().item()
            total += data.size(0)
            
        return total_loss / total, (correct / total) * 100.0


# =============================================================================
# 4. BENCHMARK UTILITIES & EXECUTION PIPELINE
# =============================================================================

def calculate_model_bandwidth(model: nn.Module) -> Tuple[int, int]:
    """Calculates full model size vs PEFT trainable payload size in Bytes."""
    full_params = sum(p.numel() for p in model.parameters())
    peft_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return full_params * 4, peft_params * 4


def get_dataset(dataset_name: str) -> Tuple[Dataset, Dataset]:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    if dataset_name.lower() == "fashionmnist":
        train_ds = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
        test_ds = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
    else:
        train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
        test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    return train_ds, test_ds


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="PEFT Federated Learning Pipeline")
    parser.add_argument("--dataset", type=str, default="mnist", choices=["mnist", "fashionmnist"])
    parser.add_argument("--num_clients", type=int, default=5)
    parser.add_argument("--alpha", type=float, default=0.5, help="Dirichlet concentration parameter")
    parser.add_argument("--rounds", type=int, default=5)
    parser.add_argument("--local_epochs", type=int, default=2)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=0.01)
    parser.add_argument("--rank", type=int, default=16, help="PEFT bottleneck adapter rank")
    args, _ = parser.parse_known_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(42)
    np.random.seed(42)

    print(f"=== Parameter-Efficient Federated Learning Aggregator [{device.upper()}] ===")
    
    # 1. Load Dataset
    train_dataset, test_dataset = get_dataset(args.dataset)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # 2. Partition Data Non-IID
    client_subsets = partition_data_dirichlet(train_dataset, num_clients=args.num_clients, alpha=args.alpha)

    # 3. Initialize Server and Models
    global_model = DeepAgentPolicyNetwork(in_channels=1, num_classes=10, adapter_rank=args.rank)
    server = FederatedServer(global_model, device=device)

    # Calculate Communication Savings
    full_bytes, peft_bytes = calculate_model_bandwidth(global_model)
    bandwidth_reduction = (1 - (peft_bytes / full_bytes)) * 100
    
    print(f"Dataset Selected: {args.dataset.upper()}")
    print(f"Full Model Payload: {full_bytes / 1024:.2f} KB")
    print(f"PEFT Uplink Payload: {peft_bytes / 1024:.2f} KB")
    print(f"Upstream Bandwidth Reduction: {bandwidth_reduction:.2f}%\n" + "-"*50)

    # Instantiate Nodes
    clients = [
        FederatedAgentNode(
            node_id=i, model_template=global_model, subset=client_subsets[i], 
            batch_size=args.batch_size, lr=args.lr, device=device
        )
        for i in range(args.num_clients)
    ]

    # 4. Federated Training Loop
    for round_num in range(1, args.rounds + 1):
        client_updates = []
        global_state = server.global_model.state_dict()

        for client in clients:
            client.local_model.load_state_dict(global_state, strict=False)
            updated_weights, n_samples = client.train_epoch(epochs=args.local_epochs)
            client_updates.append((updated_weights, n_samples))

        server.aggregate_updates(client_updates)
        val_loss, val_acc = server.evaluate(test_loader)
        print(f"Round {round_num:02d}/{args.rounds:02d} | Test Loss: {val_loss:.4f} | Test Acc: {val_acc:.2f}%")

    print("\n[Experiment Completed Successfully]")

=== Parameter-Efficient Federated Learning Aggregator [CPU] ===


100%|█████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:02<00:00, 3.43MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 381kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 1.77MB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<?, ?B/s]


Dataset Selected: MNIST
Full Model Payload: 3253.29 KB
PEFT Uplink Payload: 3253.29 KB
Upstream Bandwidth Reduction: 0.00%
--------------------------------------------------
Round 01/05 | Test Loss: 1.7773 | Test Acc: 69.54%
Round 02/05 | Test Loss: 1.4593 | Test Acc: 78.89%
Round 03/05 | Test Loss: 1.3258 | Test Acc: 81.86%
Round 04/05 | Test Loss: 1.2668 | Test Acc: 82.05%
Round 05/05 | Test Loss: 1.2651 | Test Acc: 81.48%

[Experiment Completed Successfully]
